# Milestone 2: Data Quality dan Profiling

Lanjutkan dari hasil `01_data_profiling.ipynb`. Notebook ini menerapkan profiling DuckDB `SUMMARIZE` dan transformasi Polars untuk menangani missing values, outlier, duplikat, serta validasi data dasar. Sesuaikan nama kolom dan aturan validasi dengan dataset masing-masing.

In [1]:

from pathlib import Path

import duckdb
import polars as pl

# ============================================================
# PERSIAPAN DATASET - MILESTONE 2
# ============================================================

FILENAME = "data_indonesian_religius.csv"

# Cari root project dari direktori saat ini dan parent
current_dir = Path.cwd().resolve()

possible_roots = [
    current_dir,
    *current_dir.parents
]

PROJECT_ROOT = next(
    (
        root for root in possible_roots
        if (root / "data" / "raw" / FILENAME).is_file()
    ),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f"Dataset {FILENAME} tidak ditemukan.\n"
        "Pastikan file berada di folder data/raw/."
    )

DATA_PATH = PROJECT_ROOT / "data" / "raw" / FILENAME

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = (
    PROCESSED_DIR / "indonesian_religious_corpus_bersih.parquet"
)

# ============================================================
# MEMBACA DATASET
# ============================================================

df = pl.read_csv(
    DATA_PATH,
    infer_schema_length=10_000,
    try_parse_dates=False,
    null_values=["", "NULL", "null", "None", "N/A", "NA"],
    ignore_errors=False
)

# ============================================================
# PEMETAAN KOLOM
# ============================================================

COLUMN_MAP = {
    column.lower().strip(): column
    for column in df.columns
}

def find_column(*candidates):
    for candidate in candidates:
        found = COLUMN_MAP.get(candidate.lower().strip())
        if found is not None:
            return found
    return None

COL_LABEL = find_column("Label")
COL_DENOMINATION = find_column("Denomination")
COL_LOCATION = find_column("Location")
COL_DATE = find_column("Date")
COL_TITLE = find_column("Title")
COL_TEXT = find_column("Sentence_Unit")
COL_LINK = find_column("Link", "Original_Link")

# Kolom teks wajib tersedia
if COL_TEXT is None:
    raise ValueError(
        f"Kolom Sentence_Unit tidak ditemukan. "
        f"Kolom yang tersedia: {df.columns}"
    )

# ============================================================
# INFORMASI DATASET
# ============================================================

print("=" * 60)
print("INFORMASI DATASET")
print("=" * 60)
print(f"Lokasi dataset : {DATA_PATH}")
print(f"Ukuran file    : {DATA_PATH.stat().st_size / (1024**2):,.2f} MB")
print(f"Jumlah baris   : {df.height:,}")
print(f"Jumlah kolom   : {df.width}")

print("\nPEMETAAN KOLOM")
for name, column in {
    "Label": COL_LABEL,
    "Denomination": COL_DENOMINATION,
    "Location": COL_LOCATION,
    "Date": COL_DATE,
    "Title": COL_TITLE,
    "Sentence_Unit": COL_TEXT,
    "Link": COL_LINK,
}.items():
    print(f"{name:<20}: {column}")

print("\nTIPE DATA")
for column, dtype in df.schema.items():
    print(f"{column:<25}: {dtype}")

print("\nCONTOH DATA")
display(df.head(5))


INFORMASI DATASET
Lokasi dataset : /home/jovyan/work/data/raw/data_indonesian_religius.csv
Ukuran file    : 682.73 MB
Jumlah baris   : 3,013,172
Jumlah kolom   : 7

PEMETAAN KOLOM
Label               : Label
Denomination        : Denomination
Location            : Location
Date                : Date
Title               : Title
Sentence_Unit       : Sentence_Unit
Link                : Link

TIPE DATA
Label                    : String
Denomination             : String
Location                 : String
Date                     : String
Title                    : String
Sentence_Unit            : String
Link                     : String

CONTOH DATA


Label,Denomination,Location,Date,Title,Sentence_Unit,Link
str,str,str,str,str,str,str
"""Catholic""","""Katolisitas""","""National""",null,null,"""Banyak orang yang bertanya-tan…","""https://katolisitas.org"""
"""Catholic""","""Katolisitas""","""National""",null,null,"""Sebab ketika berselingkuh, ses…","""https://katolisitas.org"""
"""Catholic""","""Katolisitas""","""National""",null,null,"""Kalian pengen masuk Surga ga?""","""https://katolisitas.org"""
"""Catholic""","""Katolisitas""","""National""",null,null,"""Tinggallah bersama kami, ya Tu…","""https://katolisitas.org"""
"""Catholic""","""Katolisitas""","""National""",null,null,"""Perjalanan dua orang murid ke …","""https://katolisitas.org"""


## 1. Profiling dengan DuckDB `SUMMARIZE`

Dapatkan ringkasan statistik seluruh kolom (tipe data, null%, min/max, distinct count) dalam satu perintah.

In [2]:
# ============================================================
# 1. PROFILING DENGAN DUCKDB SUMMARIZE
# ============================================================

connection = duckdb.connect()
connection.register("dataset", df)

profil = connection.execute(
    "SUMMARIZE SELECT * FROM dataset"
).pl()

display(profil)

# Simpan hasil profiling
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

profil.write_csv(
    OUTPUT_PATH.parent / "profiling_duckdb_summarize.csv"
)


column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,i32,i32,i32,i32,i32,i64,"decimal[9,2]"
"""Label""","""VARCHAR""","""Catholic""","""Protestant""",3,null,null,null,null,null,3013172,0.00
"""Denomination""","""VARCHAR""","""Alkitab SABDA""","""Yufid""",68,null,null,null,null,null,3013172,0.00
"""Location""","""VARCHAR""","""International""","""Sumatra""",10,null,null,null,null,null,3013172,0.00
"""Date""","""VARCHAR""",null,null,0,null,null,null,null,null,3013172,100.00
"""Title""","""VARCHAR""",null,null,0,null,null,null,null,null,3013172,100.00
"""Sentence_Unit""","""VARCHAR""","""( (21): 87) Inilah tasbih Nabi…","""𝐅𝐫𝐚𝐧𝐬 𝐒𝐢𝐧𝐮𝐫𝐚𝐭Mengucapkan terim…",1963349,null,null,null,null,null,3013172,0.00
"""Link""","""VARCHAR""","""http://Www.muslimah.or.id""","""https://zakat.ldii.or.id/zakat…",128940,null,null,null,null,null,3013172,0.00


## 2. Deteksi Missing Values

Lihat jumlah dan persentase null per kolom sebelum memutuskan strategi penanganan.

In [3]:
# ============================================================
# 2. DETEKSI MISSING VALUES
# ============================================================

jumlah_baris = len(df)

null_summary = (
    df.null_count()
    .transpose(
        include_header=True,
        header_name="kolom",
        column_names=["jumlah_null"]
    )
    .with_columns(
        (
            pl.col("jumlah_null") / max(jumlah_baris, 1) * 100
        ).round(2).alias("persen_null")
    )
    .sort("jumlah_null", descending=True)
)

display(null_summary)

# Periksa juga string kosong dan spasi saja
empty_summary = []

for column, dtype in df.schema.items():
    if dtype == pl.String:
        count_empty = df.select(
            (
                pl.col(column).is_not_null()
                & (pl.col(column).str.strip_chars() == "")
            ).sum()
        ).item()

        empty_summary.append({
            "kolom": column,
            "jumlah_string_kosong": count_empty
        })

print("Pemeriksaan string kosong:")
display(pl.DataFrame(empty_summary))


kolom,jumlah_null,persen_null
str,u32,f64
"""Date""",3013172,100.0
"""Title""",3013172,100.0
"""Label""",0,0.0
"""Denomination""",0,0.0
"""Location""",0,0.0
"""Sentence_Unit""",0,0.0
"""Link""",0,0.0


Pemeriksaan string kosong:


kolom,jumlah_string_kosong
str,i64
"""Label""",0
"""Denomination""",0
"""Location""",0
"""Date""",0
"""Title""",0
"""Sentence_Unit""",0
"""Link""",0


## 3. Penanganan Missing Values

Ganti `kolom_numerik` dan `kolom_kategorik` dengan nama kolom asli pada dataset. Pilih strategi sesuai jenis data: median/mean untuk numerik, nilai default atau modus untuk kategorik.

In [4]:

# ============================================================
# 3. PENANGANAN MISSING VALUES
# ============================================================

# Gunakan dataset asli sebagai input
df_clean = df.clone()

# Periksa jumlah missing value per kolom
missing_counts = df_clean.null_count().row(0, named=True)

print("Missing value sebelum penanganan:")
for column, count in missing_counts.items():
    percentage = count / df_clean.height * 100
    print(f"{column:<20}: {count:,} ({percentage:.2f}%)")

# Identifikasi kolom yang seluruh nilainya kosong
all_missing_columns = [
    column
    for column in df_clean.columns
    if df_clean[column].null_count() == df_clean.height
]

# Hapus hanya kolom yang 100% kosong
df_clean = df_clean.drop(all_missing_columns)

print("\nKolom yang dihapus:", all_missing_columns)
print(f"Jumlah baris awal  : {df.height:,}")
print(f"Jumlah baris akhir : {df_clean.height:,}")
print(f"Jumlah kolom akhir : {df_clean.width}")

# Verifikasi hasil
print("\nMissing value setelah penanganan:")
display(df_clean.null_count().transpose(
    include_header=True,
    header_name="kolom",
    column_names=["jumlah_missing"]
))


Missing value sebelum penanganan:
Label               : 0 (0.00%)
Denomination        : 0 (0.00%)
Location            : 0 (0.00%)
Date                : 3,013,172 (100.00%)
Title               : 3,013,172 (100.00%)
Sentence_Unit       : 0 (0.00%)
Link                : 0 (0.00%)

Kolom yang dihapus: ['Date', 'Title']
Jumlah baris awal  : 3,013,172
Jumlah baris akhir : 3,013,172
Jumlah kolom akhir : 5

Missing value setelah penanganan:


kolom,jumlah_missing
str,u32
"""Label""",0
"""Denomination""",0
"""Location""",0
"""Sentence_Unit""",0
"""Link""",0


## 4. Deteksi dan Penanganan Outlier

Gunakan metode IQR sebagai titik awal, lalu kombinasikan dengan batas bisnis (domain knowledge) yang relevan dengan dataset Anda.

In [11]:

# ============================================================
# 4. DETEKSI OUTLIER PANJANG TEKS
# ============================================================

# Gunakan dataset hasil penanganan missing values
if "df_clean" not in globals():
    raise NameError(
        "Variabel df_clean belum tersedia. "
        "Jalankan dahulu cell penanganan missing values."
    )

if COL_TEXT is None or COL_TEXT not in df_clean.columns:
    raise ValueError(
        f"Kolom teks tidak ditemukan. Kolom tersedia: {df_clean.columns}"
    )

# Hitung panjang teks setiap baris
df_outlier_handled = df_clean.with_columns(
    pl.col(COL_TEXT)
    .str.len_chars()
    .alias("_panjang_teks")
)

# Hitung kuartil dan IQR
q1 = df_outlier_handled["_panjang_teks"].quantile(0.25)
q3 = df_outlier_handled["_panjang_teks"].quantile(0.75)

iqr = q3 - q1
batas_bawah = max(0, q1 - 1.5 * iqr)
batas_atas = q3 + 1.5 * iqr

# Tandai outlier tanpa langsung menghapusnya
kondisi_outlier = (
    (pl.col("_panjang_teks") < batas_bawah)
    | (pl.col("_panjang_teks") > batas_atas)
)

outlier_summary = df_outlier_handled.select(
    pl.len().alias("total_baris"),
    kondisi_outlier.sum().alias("jumlah_outlier")
)

print("=" * 60)
print("HASIL DETEKSI OUTLIER PANJANG TEKS")
print("=" * 60)
print(f"Q1 panjang teks : {q1}")
print(f"Q3 panjang teks : {q3}")
print(f"IQR             : {iqr}")
print(f"Batas bawah     : {batas_bawah}")
print(f"Batas atas      : {batas_atas}")

display(outlier_summary)

# Tampilkan maksimal 10 contoh outlier
outlier_examples = (
    df_outlier_handled
    .filter(kondisi_outlier)
    .sort("_panjang_teks", descending=True)
)

display(outlier_examples.head(10))


HASIL DETEKSI OUTLIER PANJANG TEKS
Q1 panjang teks : 75.0
Q3 panjang teks : 154.0
IQR             : 79.0
Batas bawah     : 0
Batas atas      : 272.5


total_baris,jumlah_outlier
u32,u32
3013172,104083


Label,Denomination,Location,Sentence_Unit,Link,_panjang_teks
str,str,str,str,str,u32
"""Catholic""","""Dokpen KWI""","""National""","""Tahapan pertumbuhan yang diusu…","""https://www.dokpenkwi.org/sdg-…",500
"""Catholic""","""Katolisitas""","""National""","""Pada saat Rasul Paulus memperi…","""https://katolisitas.org/tak-pe…",500
"""Catholic""","""Katolisitas""","""National""","""Berikut ini adalah tanggapan d…","""https://katolisitas.org/diskus…",500
"""Catholic""","""Dokpen KWI""","""National""","""Dalam mengambil inisiatif dan …","""https://www.dokpenkwi.org/sdg-…",500
"""Catholic""","""Dokpen KWI""","""National""","""Tahapan pertumbuhan yang diusu…","""https://www.dokpenkwi.org/?p=1…",500
"""Catholic""","""Dokpen KWI""","""National""","""Para Uskup harus melakukan seg…","""https://www.dokpenkwi.org/anti…",500
"""Catholic""","""Keuskupan Bogor""","""Java""","""Dua hal yang disepekati itu te…","""https://keuskupanbogor.org/201…",500
"""Catholic""","""KAS - Keuskupan Agung Semarang""","""Java""","""Para Uskup harus melakukan seg…","""https://kas.or.id/antiquum-min…",500
"""Catholic""","""Dokpen KWI""","""National""","""Dalam mengambil inisiatif dan …","""https://www.dokpenkwi.org/?p=4…",500


## 5. Deduplikasi

Tentukan kolom kunci bisnis (misalnya ID unik) sebelum menghapus duplikat — jangan deduplikasi berdasarkan seluruh baris.

In [12]:
# ============================================================
# 5. DEDUPLIKASI
# ============================================================

candidate_keys = [
    COL_TEXT,
    COL_LABEL,
    COL_DENOMINATION,
    COL_LINK,
]

KOLOM_KUNCI = [
    column for column in candidate_keys
    if column is not None
]

if not KOLOM_KUNCI:
    raise ValueError("Tidak ada kolom kunci untuk deduplikasi.")

total_baris = len(df_outlier_handled)

df_dedup = df_outlier_handled.unique(
    subset=KOLOM_KUNCI,
    keep="first",
    maintain_order=True
)

print("Kolom kunci:", KOLOM_KUNCI)
print(f"Baris sebelum deduplikasi: {total_baris:,}")
print(f"Baris sesudah deduplikasi: {len(df_dedup):,}")
print(f"Duplikat dihapus: {total_baris - len(df_dedup):,}")

# Hapus kolom bantu panjang teks
if "_panjang_teks" in df_dedup.columns:
    df_dedup = df_dedup.drop("_panjang_teks")

Kolom kunci: ['Sentence_Unit', 'Label', 'Denomination', 'Link']
Baris sebelum deduplikasi: 3,013,172
Baris sesudah deduplikasi: 3,012,817
Duplikat dihapus: 355


## 6. Validasi Data Dasar

Tulis beberapa aturan validasi sederhana untuk memastikan data bersih memenuhi batas yang masuk akal. Gantikan contoh aturan di bawah dengan aturan yang relevan dengan dataset Anda.

In [13]:
# ============================================================
# 6. VALIDASI DATA DASAR
# ============================================================

connection.register("dataset_bersih", df_dedup)

# Validasi jumlah baris, kolom penting kosong,
# dan duplikat berdasarkan kunci korpus.
validasi_data = {
    "total_baris": len(df_dedup),
    "total_kolom": len(df_dedup.columns),
    "baris_dengan_teks_kosong": 0,
    "baris_dengan_label_kosong": 0,
    "duplikat_kunci": 0,
}

if COL_TEXT is not None:
    validasi_data["baris_dengan_teks_kosong"] = df_dedup.filter(
        pl.col(COL_TEXT).is_null()
        | (pl.col(COL_TEXT).str.strip_chars() == "")
    ).height

if COL_LABEL is not None:
    validasi_data["baris_dengan_label_kosong"] = df_dedup.filter(
        pl.col(COL_LABEL).is_null()
        | (pl.col(COL_LABEL).cast(pl.String).str.strip_chars() == "")
    ).height

validasi_data["duplikat_kunci"] = (
    len(df_dedup)
    - df_dedup.select(KOLOM_KUNCI).unique().height
)

validasi = pl.DataFrame({
    "indikator": list(validasi_data.keys()),
    "hasil": list(validasi_data.values())
})

display(validasi)

assert validasi_data["baris_dengan_teks_kosong"] == 0, (
    "Masih terdapat teks kosong."
)

if COL_LABEL is not None:
    assert validasi_data["baris_dengan_label_kosong"] == 0, (
        "Masih terdapat label kosong."
    )

assert validasi_data["duplikat_kunci"] == 0, (
    "Masih terdapat duplikat berdasarkan kolom kunci."
)

print("Validasi data selesai.")

indikator,hasil
str,i64
"""total_baris""",3012817
"""total_kolom""",5
"""baris_dengan_teks_kosong""",0
"""baris_dengan_label_kosong""",0
"""duplikat_kunci""",0


Validasi data selesai.


## 7. Simpan Data Bersih

Simpan hasil ke `data/processed/` agar dapat dipakai pada Milestone 3 tanpa mengulang proses cleaning.

In [17]:

# ============================================================
# 7. SIMPAN DATA BERSIH
# ============================================================

# Pastikan lokasi output tersedia
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

# Simpan dataset hasil cleaning
df_dedup.write_parquet(OUTPUT_PATH)

# ============================================================
# RINGKASAN MILESTONE 2
# ============================================================

print("=" * 60)
print("RINGKASAN MILESTONE 2")
print("=" * 60)

print(f"Baris awal          : {len(df):,}")
print(f"Baris setelah missing: {len(df_clean):,}")
print(f"Baris setelah dedup : {len(df_dedup):,}")
print(f"Baris berubah       : {len(df) - len(df_dedup):,}")
print(f"Kolom akhir         : {len(df_dedup.columns)}")
print(f"Nama kolom          : {df_dedup.columns}")
print(f"File tersimpan      : {OUTPUT_PATH}")

# Ukuran file setelah penyimpanan
ukuran_mb = OUTPUT_PATH.stat().st_size / (1024 ** 2)
print(f"Ukuran akhir        : {ukuran_mb:.2f} MB")

print("=" * 60)
print("MILESTONE 2 SELESAI")
print("=" * 60)


RINGKASAN MILESTONE 2
Baris awal          : 3,013,172
Baris setelah missing: 3,013,172
Baris setelah dedup : 3,012,817
Baris berubah       : 355
Kolom akhir         : 5
Nama kolom          : ['Label', 'Denomination', 'Location', 'Sentence_Unit', 'Link']
File tersimpan      : /home/jovyan/work/data/processed/indonesian_religious_corpus_bersih.parquet
Ukuran akhir        : 120.88 MB
MILESTONE 2 SELESAI


## Catatan Milestone 2

Dokumentasikan di sini: dimensi kualitas data yang paling bermasalah, strategi yang dipilih untuk missing values dan outlier, jumlah baris yang berubah, serta aturan validasi yang diterapkan.